# 02 — MultiROCKET

MultiROCKET feature extraction and persistence. Uses `src.features.multirocket` and `src.features.memmap`.

In [ ]:
# Path fix: ensure we use this repo's src (not e.g. OneDrive)
from pathlib import Path
import sys
PROJECT_ROOT = Path.cwd().resolve().parents[0]
project_root_str = str(PROJECT_ROOT)
if project_root_str in sys.path:
    sys.path.remove(project_root_str)
sys.path.insert(0, project_root_str)
for name in list(sys.modules.keys()):
    if name == "src" or name.startswith("src."):
        del sys.modules[name]

import logging
from src.utils import SEED, set_global_seed
from src.utils.paths import get_multirocket_features_dir, get_models_dir, get_splits_dir, ensure_dir
from src.data import load_raw_dataset, create_splits, load_splits, save_splits
from src.features.multirocket import (
    create_multirocket_transformer,
    fit_multirocket,
    transform_multirocket_batched,
    save_multirocket_transformer,
)
from src.features.memmap import write_array_to_memmap, open_memmap_read

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
print("Using PROJECT_ROOT:", PROJECT_ROOT)
print("SEED:", SEED)

In [ ]:
# Constants: 60/20/20 splits, MultiROCKET params
TEST_SIZE = 0.2
VAL_SIZE = 0.2
TASK_TYPE = "multilabel"
NUM_KERNELS = 2048
BATCH_SIZE = 1024
print("Split: test_size=%s val_size=%s (60/20/20) | num_kernels=%s batch_size=%s" % (TEST_SIZE, VAL_SIZE, NUM_KERNELS, BATCH_SIZE))

In [ ]:
# Load data and set seed
print("Loading dataset from cache...")
X, y = load_raw_dataset()
print("Loaded X shape %s, y shape %s" % (X.shape, y.shape))

set_global_seed(SEED)
print("Global seed set to %s" % SEED)

In [ ]:
# Splits: load existing or create 60/20/20
splits_path = get_splits_dir() / "splits.npz"
if splits_path.exists():
    print("Loading splits (60/20/20) from %s..." % splits_path)
    idx_train, idx_val, idx_test = load_splits(splits_path)
else:
    print("Creating splits (60/20/20)...")
    idx_train, idx_val, idx_test = create_splits(y, task_type=TASK_TYPE, test_size=TEST_SIZE, val_size=VAL_SIZE, seed=SEED)
    save_splits(idx_train, idx_val, idx_test, splits_path)
print("Train %d, val %d, test %d" % (len(idx_train), len(idx_val), len(idx_test)))

In [ ]:
# Create and fit MultiROCKET on training data only
print("Fitting MultiROCKET on %d samples..." % len(idx_train))
transformer = create_multirocket_transformer(num_kernels=NUM_KERNELS, seed=SEED)
X_train = X[idx_train]
fit_multirocket(transformer, X_train)
print("Fit complete.")

In [ ]:
# Transform train/val/test in batches (progress logged per batch)
print("Transforming train set...")
F_train = transform_multirocket_batched(transformer, X[idx_train], batch_size=BATCH_SIZE)
print("Transforming val set...")
F_val = transform_multirocket_batched(transformer, X[idx_val], batch_size=BATCH_SIZE) if len(idx_val) > 0 else None
print("Transforming test set...")
F_test = transform_multirocket_batched(transformer, X[idx_test], batch_size=BATCH_SIZE)
print("F_train %s, F_val %s, F_test %s" % (F_train.shape, F_val.shape if F_val is not None else None, F_test.shape))

In [ ]:
# Save transformer
mr_transformer_path = get_models_dir() / "multirocket" / "transformer.joblib"
ensure_dir(mr_transformer_path.parent)
print("Saving transformer to %s..." % mr_transformer_path)
save_multirocket_transformer(transformer, mr_transformer_path)
print("Saved.")

In [ ]:
# Save features (memmap) and shape metadata
mr_dir = ensure_dir(get_multirocket_features_dir())
print("Writing train.dat (shape %s)..." % (F_train.shape,))
write_array_to_memmap(mr_dir / "train.dat", F_train)
print("Writing test.dat (shape %s)..." % (F_test.shape,))
write_array_to_memmap(mr_dir / "test.dat", F_test)
if F_val is not None:
    print("Writing val.dat (shape %s)..." % (F_val.shape,))
    write_array_to_memmap(mr_dir / "val.dat", F_val)

# Persist shapes for reopening memmaps
import numpy as np
shapes = {"train": np.array(F_train.shape), "test": np.array(F_test.shape)}
if F_val is not None:
    shapes["val"] = np.array(F_val.shape)
np.savez(mr_dir / "shapes.npz", **shapes)
print("Saved shapes to %s" % (mr_dir / "shapes.npz"))
print("Done. Features in %s" % mr_dir)